# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
#Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
#your code here

spaceship = spaceship.dropna()


spaceship['CabinDeck'] = spaceship['Cabin'].astype(str).str[0].str.upper()


target_col = 'Transported'
drop_cols = [c for c in ['PassengerId', 'Name', 'Cabin'] if c in spaceship.columns]

y = spaceship[target_col]
X = pd.get_dummies(
    spaceship.drop(columns=drop_cols + [target_col]),
    drop_first=False
)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


k = min(30, X.shape[1])
selector = SelectKBest(score_func=f_classif, k=k)
X_fs = selector.fit_transform(X_scaled, y)

X.shape, X_fs.shape, y.shape


((6606, 24), (6606, 24), (6606,))

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [4]:
#your code here
# Split de train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_fs, y, test_size=0.20, random_state=42, stratify=y
)

# Modelo baseline: Random Forest (el más razonable para mejorar con tuning)
rf_base = RandomForestClassifier(
    random_state=42,
    n_estimators=200
)
rf_base.fit(X_train, y_train)
rf_base


RandomForestClassifier(n_estimators=200, random_state=42)

- Evaluate your model

In [5]:
#your code here
y_pred_base = rf_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)
f1_base  = f1_score(y_test, y_pred_base, average='macro')

print("Baseline RandomForest")
print("Accuracy:", acc_base)
print("F1-macro:", f1_base)
print("\nClassification report:\n")
print(classification_report(y_test, y_pred_base))


Baseline RandomForest
Accuracy: 0.7829046898638427
F1-macro: 0.7829015843479763

Classification report:

              precision    recall  f1-score   support

       False       0.78      0.79      0.78       656
        True       0.79      0.78      0.78       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [6]:
#your code here
# Definición de la malla de hiperparámetros para Grid Search
param_grid = {
    'n_estimators': [100, 300, 600],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}
param_grid


{'n_estimators': [100, 300, 600],
 'max_depth': [None, 10, 20],
 'min_samples_split': [2, 5],
 'min_samples_leaf': [1, 2],
 'max_features': ['sqrt', 'log2']}

- Run Grid Search

In [7]:
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='accuracy',   # puedes cambiar a 'f1_macro' si prefieres
    cv=5,
    n_jobs=-1,
    verbose=0
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score (accuracy):", grid.best_score_)
best_model = grid.best_estimator_
best_model


Best params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 600}
Best CV score (accuracy): 0.8016681717840658


RandomForestClassifier(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=600, random_state=42)

- Evaluate your model

In [8]:
y_pred_tuned = best_model.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned  = f1_score(y_test, y_pred_tuned, average='macro')

print("Tuned RandomForest (GridSearchCV)")
print("Accuracy:", acc_tuned)
print("F1-macro:", f1_tuned)
print("\nClassification report:\n")
print(classification_report(y_test, y_pred_tuned))

# Comparativa simple
print("\nComparativa:")
print(f"- Baseline -> Acc: {acc_base:.4f} | F1-macro: {f1_base:.4f}")
print(f"- Tuned    -> Acc: {acc_tuned:.4f} | F1-macro: {f1_tuned:.4f}")


Tuned RandomForest (GridSearchCV)
Accuracy: 0.7912254160363086
F1-macro: 0.7912134695671281

Classification report:

              precision    recall  f1-score   support

       False       0.79      0.79      0.79       656
        True       0.79      0.79      0.79       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322


Comparativa:
- Baseline -> Acc: 0.7829 | F1-macro: 0.7829
- Tuned    -> Acc: 0.7912 | F1-macro: 0.7912
